

Video link- https://drive.google.com/file/d/1uX4QdqALxz-TvaQyVmjPPaSqAHLhD0x8/view?usp=sharing

Github link-https://github.com/Mkyadav11/CS-432-Databases/tree/main/Assignment%203




# **CS432 - Databases Track 1 Assignment 3**

## Module A: ACID Validation

### 1.OBJECTIVE:
The objective of this module is to extend an existing B+ Tree-based mini database system to support transaction management, failure recovery, and ACID guarantees. The focus is on ensuring correctness, reliability, and robustness under concurrent execution and system failures.

The implementation builds upon a previously developed system that already supports table abstraction and B+ Tree-based storage.

# 2.System Architecture

The system is designed as a modular mini-database engine where each component is responsible for a specific aspect of transaction processing and data management. The architecture consists of the following core components:



## 2.1 B+ Tree Storage Engine

The B+ Tree serves as the primary storage mechanism for all relations in the system. Each table is implemented as an independent B+ Tree, where:

- The **primary key** is used as the index key  
- The **value** represents the complete record  

All database operations such as insert, update, delete, and search are performed directly on the B+ Tree. Leaf nodes store actual records, while internal nodes maintain indexing structure for efficient traversal.

The B+ Tree also supports:

- Ordered data access through linked leaf nodes  
- Range queries and full table scans  
- Snapshot creation for recovery purposes  

Thus, the B+ Tree acts as both the storage engine and indexing structure, ensuring efficient and consistent data access.



## 2.2 Transaction Manager

The Transaction Manager coordinates all transactional operations and ensures ACID properties. It provides support for:

- **BEGIN**: Initializes a new transaction with a unique transaction ID  
- **COMMIT**: Finalizes the transaction and makes changes permanent  
- **ROLLBACK**: Reverts all changes made during the transaction  

Each transaction maintains:

- An **undo log** to store previous states of modified records  
- A list of **acquired locks** to ensure isolation  

Before any modification (insert, update, delete), the old value is stored in the undo log. In case of rollback, all operations are reversed in the correct order using this log.



## 2.3 Lock Manager (Concurrency Control)

The system implements a **table-level locking mechanism** to ensure isolation between concurrent transactions.

- Each table can be locked by only one transaction at a time  
- Other transactions attempting to access the same table must wait  
- Locks are released after commit or rollback  

A condition-variable-based mechanism is used to block and wake up waiting transactions. This ensures that:

- No dirty reads occur  
- Transactions execute in a serializable manner  



## 2.4 Write-Ahead Log (WAL)

The Write-Ahead Log is responsible for ensuring **atomicity and durability**.

- Every operation is logged before being applied to the database  
- Logs are written to disk using `fsync` to guarantee persistence  

The WAL records include:

- Transaction start (**BEGIN**)  
- Data modifications (**UPDATE**)  
- Transaction completion (**COMMIT / ROLLBACK**)  


## 2.5 Recovery Mechanism

The system supports crash recovery using a simplified ARIES-like approach:

- **REDO Phase**: Reapply all committed transactions  
- **UNDO Phase**: Reverse all incomplete transactions  

During recovery:

- Committed transactions are reapplied to ensure durability  
- Incomplete transactions are rolled back using logged old values  

This guarantees that the database returns to a consistent state after failure.



## 2.6 Overall Workflow

1. A transaction begins using the Transaction Manager  
2. Required locks are acquired on tables  
3. Operations are logged in WAL before execution  
4. Changes are applied to the B+ Tree  
5. On **COMMIT**:
   - Commit is logged  
   - Locks are released  
6. On failure:
   - Undo log is used to rollback changes  
   - Recovery mechanism ensures consistency  


# 3.Transaction Model

The system supports transactions across multiple relations using the following operations:

- **BEGIN**
- **COMMIT**
- **ROLLBACK**

Each transaction is assigned a unique transaction ID and operates on multiple tables. The Transaction Manager ensures that all operations within a transaction are executed as a single atomic unit.



## 3.1 Multi-Relation Transaction

Transactions in the system span multiple tables such as:

- `member`
- `meallog`
- `monthly_mess_payment`

A typical transaction involves:

- Updating a payment record  
- Inserting a meal log entry  
- Inserting or updating a member  

All operations are executed within a single transaction context.



## 3.2 Atomic Execution

The system ensures that:

- Either all operations in a transaction are successfully applied  
- Or all operations are completely rolled back in case of failure  

This guarantees that no partial updates remain in the database.

# 4. ACID Properties Implementation

The system ensures reliability and correctness of transactions by implementing the four fundamental ACID properties: Atomicity, Consistency, Isolation, and Durability. These properties are achieved through a combination of undo logging, table-level locking, and Write-Ahead Logging (WAL).



## 4.1 Atomicity

Atomicity ensures that a transaction is either fully completed or fully rolled back.

In the implementation, each transaction maintains an **undo log**, which stores the previous state of every modified record in the format:

(table, key, old_value)

Before applying any modification (insert, update, delete), the old value is recorded in the undo log and the change is written to the WAL.

In case of failure, the system performs a rollback by applying all undo operations in reverse order. This restores the database to its original state.

A simulated crash during execution demonstrates that even if some operations are completed, all changes are undone, ensuring that no partial updates remain.


## 4.2 Consistency

Consistency ensures that the database always remains in a valid state before and after a transaction.

The system enforces application-level constraints, including:

- Valid roles for members  
- Non-zero payment amounts  
- Valid foreign key references between tables  

If any constraint is violated during a transaction, an exception is raised and the transaction is rolled back.

Since all operations directly modify the B+ Tree (which represents the database state), consistency is maintained at all times. No invalid or partially updated records are ever stored.



## 4.3 Isolation

Isolation ensures that concurrent transactions do not interfere with each other.

The system implements **table-level exclusive locking** using a lock manager:

- A transaction must acquire a lock before accessing a table  
- If another transaction already holds the lock, it is blocked until the lock is released  
- Locks are released only after COMMIT or ROLLBACK  

This guarantees that:

- Uncommitted changes are never visible to other transactions  
- Dirty reads are prevented  
- Transactions execute in a serializable manner  

Experimental validation using concurrent threads shows that a reader transaction waits while a writer holds a lock and only reads committed data.



## 4.4 Durability

Durability ensures that once a transaction is committed, its changes persist even after system failure.

The system uses **Write-Ahead Logging (WAL)** to achieve durability:

- Every operation is logged before being applied to the database  
- Log records are flushed to disk using `fsync`, ensuring persistence  

During recovery:

- All committed transactions are **redone**  
- All incomplete transactions are **undone**  

A simulated system restart demonstrates that committed data is successfully recovered from the log, confirming that no data is lost after failure.



# 5. Failure Handling and Recovery

The system is designed to handle failures during transaction execution and ensure that the database remains in a consistent state using Write-Ahead Logging (WAL).

All operations are first recorded in the WAL before being applied to the database. This guarantees that sufficient information is available to recover from failures.

## 5.1 Failure Handling

During execution, the system may encounter failures such as crashes in the middle of a transaction. In such cases:

- **Incomplete transactions are rolled back** using stored old values  
- **Committed transactions are preserved** and not affected  

Rollback is performed using the transaction’s undo log, which restores the previous state of all modified records.


## 5.2 Recovery Mechanism

After a system restart, the database state is reconstructed using the WAL.

The recovery process works as follows:

- Transactions with a **COMMIT record** are identified as completed  
- Transactions without a commit are treated as incomplete  

Then:

- All **committed transactions are redone** to ensure durability  
- All **incomplete transactions are undone** to remove partial updates  



## 5.3 Correctness Guarantee

This approach ensures that:

- No partial updates remain after a crash  
- All committed data is preserved  
- The database always returns to a consistent and valid state  

Thus, the system successfully maintains reliability under failure conditions.

# 6. Multi-User Conflict Handling

The system supports concurrent transactions and handles conflicts using a table-level locking mechanism.



## 6.1 Locking Mechanism

Each transaction must acquire a lock before accessing a table:

- When a transaction performs an operation (insert, update, delete, select), it requests a lock on the corresponding table  
- If the table is already locked by another transaction, the requesting transaction is blocked  
- The transaction resumes only after the lock is released  

Locks are released only after the transaction completes (COMMIT or ROLLBACK).


## 6.2 Conflict Resolution

Conflicts occur when multiple transactions attempt to access the same table simultaneously. The system resolves these conflicts as follows:

- Only one transaction can hold a lock on a table at a time  
- Other transactions must wait until the current transaction finishes  
- A timeout mechanism prevents indefinite waiting  

This ensures controlled access to shared data.



## 6.3 Isolation Guarantee

The locking mechanism ensures that:

- Uncommitted changes are not visible to other transactions  
- Dirty reads are prevented  
- Transactions execute in a serializable manner  

Thus, concurrent execution does not lead to data inconsistency or corruption.



## 6.4 Implementation Details

The system uses a dedicated **Lock Manager**:

- Maintains a mapping of tables to the transaction holding the lock  
- Uses condition variables to block and wake up waiting transactions  
- Supports safe acquisition and release of locks  

Each transaction keeps track of acquired locks and releases them after completion.



## 6.5 Summary

Through table-level locking, the system ensures safe concurrent execution by preventing conflicts and maintaining data correctness.

# 7. Experiments Performed

The system was tested using structured experiments implemented in the main execution script. Each experiment validates one of the ACID properties.



## 7.1 Atomicity Test

A transaction involving multiple tables was executed and a crash was simulated during execution.

- Payment status was updated  
- Meal log entry was inserted  
- A crash was triggered before completion  

After rollback, all changes were undone and the database state remained unchanged.



## 7.2 Consistency Test

Constraint violations were tested to ensure database validity:

- Insertion of a member with an invalid role  
- Insertion of a payment with zero amount  
- Insertion of a record with an invalid foreign key  

All invalid operations were rejected and rolled back successfully.



## 7.3 Isolation Test

Concurrent transactions were executed using multiple threads:

- A writer transaction updated a record and held a lock  
- A reader transaction attempted to access the same data  

The reader was blocked until the writer released the lock and only observed committed data.



## 7.4 Durability Test

A transaction was committed and the system was restarted.

After restart, all committed data was successfully recovered using WAL, confirming persistence.



All experiments passed successfully, validating the correctness of the system.

# 8. Observations

The implementation and experimental evaluation of the system lead to the following observations:

- **Write-Ahead Logging (WAL) ensures reliable recovery**  
  The use of WAL guarantees that all operations are recorded before execution. This allows the system to correctly recover from failures by redoing committed transactions and undoing incomplete ones.

- **Locking mechanism prevents inconsistent reads**  
  The table-level locking strategy ensures that concurrent transactions do not interfere with each other. Readers are blocked when a writer holds a lock, preventing dirty reads and maintaining isolation.

- **Multi-relation transactions execute correctly**  
  Transactions involving multiple tables behave as expected. All operations within a transaction are applied atomically, and in case of failure, all changes are rolled back successfully.

- **System restores correct state after restart**  
  The recovery mechanism successfully reconstructs the database state after a simulated restart. All committed data persists, while incomplete transactions are removed.

Overall, the system demonstrates correct behavior under concurrent execution and failure conditions, validating the effectiveness of the implemented ACID mechanisms.

# 9. Limitations

While the system successfully demonstrates ACID-compliant transaction processing, it has the following limitations:

- **Basic locking mechanism**  
  The system uses table-level locking, which limits concurrency. More advanced techniques such as row-level locking or Multi-Version Concurrency Control (MVCC) are not implemented.

- **No support for distributed transactions**  
  The implementation is limited to a single-node system and does not handle distributed or networked databases.

- **Simplified recovery mechanism**  
  The recovery process is a simplified version of ARIES and does not include advanced features such as checkpoints or log compaction.

- **Performance overhead due to logging**  
  Frequent disk writes (`fsync`) for every log entry can impact performance, especially under high transaction loads.

- **Scalability constraints**  
  The system is designed for demonstration purposes and may not scale efficiently for very large datasets or high concurrency workloads.



Overall, while the system effectively demonstrates core database concepts, it can be further enhanced for performance, scalability, and advanced concurrency control.